<a href="https://colab.research.google.com/github/kipkii/kipki-s-/blob/main/I_love_Stake_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# 축평원 데이터 불러오기 (파일명은 태연님이 저장하신 파일명으로 맞춰주세요)
df_cattle = pd.read_csv("/content/Cattle_Grade_20180926.csv", encoding="utf-8-sig")

# 🚨 범인 색출: 파이썬이 인식한 진짜 컬럼명 리스트 출력
print(df_cattle.columns)

Index(['경매일자', '시장코드', '성별', '등급', '경락두수', '평균도체중', '평균가격'], dtype='object')


In [3]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 데이터 불러오기 (수집하신 파일명으로 수정해주세요)
# ==========================================
print("데이터를 불러오는 중...")
df_cattle = pd.read_csv("/content/Cattle_Grade_20180926.csv", encoding="utf-8-sig") # 혹은 cp949
df_weather = pd.read_csv("/content/Weather_2016_to_2018.csv", encoding="utf-8-sig")

# ==========================================
# 2. 날짜(Date) 컬럼 통일화 전처리
# ==========================================
# 축평원 데이터: '20160101' (정수/문자열) -> Datetime 객체로 변환
df_cattle['Date'] = pd.to_datetime(df_cattle['경매일자'].astype(str))

# 기상청 데이터: '2016-01-01' (문자열) -> Datetime 객체로 변환
df_weather['Date'] = pd.to_datetime(df_weather['일자'])

# ==========================================
# 3. 두 데이터 완벽하게 결합 (Merge)
# ==========================================
# 'Date'를 기준으로 교집합(inner) 병합
df_merged = pd.merge(df_cattle, df_weather, on='Date', how='inner')

print(f"✅ 병합 완료! 총 {len(df_merged)}건의 데이터 매핑 성공")

# ==========================================
# 4. [핵심] 가축 열스트레스 지수 (THI) 파생변수 계산
# ==========================================
# 💡 THI 공식: THI = (1.8 × T + 32) - [(0.55 - 0.0055 × RH) × (1.8 × T - 26)]
# T: 평균기온, RH: 평균상대습도

T = df_merged['평균기온']
RH = df_merged['평균상대습도']

df_merged['THI'] = (1.8 * T + 32) - ((0.55 - 0.0055 * RH) * (1.8 * T - 26))

# 분석 편의를 위해 소수점 둘째 자리까지 반올림
df_merged['THI'] = df_merged['THI'].round(2)

# THI 기준 위험도 라벨링 (한우 기준 일반적인 컷오프)
# 72 미만: 쾌적 / 72~79: 약한 스트레스 / 80~89: 강한 스트레스 / 90 이상: 폐사 위험
conditions = [
    (df_merged['THI'] < 72),
    (df_merged['THI'] >= 72) & (df_merged['THI'] < 80),
    (df_merged['THI'] >= 80) & (df_merged['THI'] < 90),
    (df_merged['THI'] >= 90)
]
choices = ['쾌적(안전)', '주의(경미)', '경고(위험)', '위험(심각)']
df_merged['THI_Status'] = np.select(conditions, choices, default='알수없음')

# ==========================================
# 5. 불필요한 중복 컬럼 정리 및 마트 저장
# ==========================================
cols_to_drop = ['경매일자', '일자', '지점코드']
df_merged = df_merged.drop(columns=cols_to_drop)

# 보기 좋게 Date를 맨 앞으로 이동
cols = ['Date'] + [c for c in df_merged if c != 'Date']
df_merged = df_merged[cols]

# 파일 저장
df_merged.to_csv("BigWave_Analysis_Mart_Pilot.csv", index=False, encoding="utf-8-sig")

print("\n🎉 최종 분석용 데이터 마트 구축 완료!")
print(df_merged[['Date', '평균기온', '평균상대습도', 'THI', 'THI_Status', '성별', '등급', '평균도체중']].head())

데이터를 불러오는 중...
✅ 병합 완료! 총 36855건의 데이터 매핑 성공

🎉 최종 분석용 데이터 마트 구축 완료!
        Date  평균기온  평균상대습도    THI THI_Status  성별    등급  평균도체중
0 2016-01-01   0.1    78.8  35.19     쾌적(안전)  전체  1++A    NaN
1 2016-01-01   0.1    78.8  35.19     쾌적(안전)  전체  1++B    NaN
2 2016-01-01   0.1    78.8  35.19     쾌적(안전)  전체    등외    NaN
3 2016-01-01   0.1    78.8  35.19     쾌적(안전)  전체    3C    NaN
4 2016-01-01   0.1    78.8  35.19     쾌적(안전)  전체    3B    NaN
